# Lesson 5 — Look back, never forward (the causal mask)

In lesson 4 every token could attend to every position, including later ones. For next-character prediction that's
cheating: position t would be able to read the answer at t+1. The fix is one line — blank out the future
**before** the softmax.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

## Same embedded batch as lesson 4

In [2]:
# --- rebuild the embedded batch from Lessons 2-3 ---
text = open("shakespeare.txt").read()
chars = sorted(set(text)); vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
block_size, batch_size, n_embd = 8, 4, 32

def get_batch(src, block, bs):
    ix = torch.randint(len(src) - block, (bs,))
    return (torch.stack([src[i:i + block] for i in ix]),
            torch.stack([src[i + 1:i + block + 1] for i in ix]))

xb, yb = get_batch(data, block_size, batch_size)
x = nn.Embedding(vocab_size, n_embd)(xb) + nn.Embedding(block_size, n_embd)(torch.arange(block_size))
B, T, C = x.shape

## Raw attention scores (no mask yet)
Same head as lesson 4. First I print the unmasked weights so I can see the "future" entries above the diagonal.

In [3]:
head_size = 16
key, query, value = (nn.Linear(C, head_size, bias=False) for _ in range(3))
k, q, v = key(x), query(x), value(x)
wei = q @ k.transpose(-2, -1) * head_size ** -0.5

print("BEFORE masking — token attends everywhere (note non-zero above the diagonal):")
print(F.softmax(wei, dim=-1)[0].round(decimals=2))

BEFORE masking — token attends everywhere (note non-zero above the diagonal):
tensor([[0.1900, 0.1200, 0.0800, 0.2300, 0.0700, 0.1400, 0.0700, 0.1100],
        [0.1100, 0.2100, 0.0800, 0.0300, 0.2300, 0.2000, 0.0600, 0.1000],
        [0.1300, 0.1100, 0.0100, 0.0700, 0.2500, 0.0900, 0.2000, 0.1400],
        [0.1000, 0.1000, 0.1500, 0.2300, 0.1400, 0.1000, 0.1100, 0.0700],
        [0.1500, 0.1100, 0.0700, 0.0700, 0.1300, 0.2700, 0.0400, 0.1500],
        [0.1300, 0.0900, 0.2200, 0.0800, 0.1400, 0.1300, 0.1100, 0.1000],
        [0.1100, 0.1100, 0.2400, 0.0800, 0.1000, 0.0600, 0.1900, 0.1100],
        [0.0800, 0.2600, 0.0800, 0.0900, 0.2300, 0.0900, 0.1000, 0.0600]],
       grad_fn=<RoundBackward1>)


## Apply the mask
- `torch.tril(ones)` is 1 on and below the diagonal = "allowed to look at".
- Wherever it's 0 (the future), set the score to `-inf`.
- Softmax then gives those spots `exp(-inf) = 0`, and each row re-normalises over only the past + current token.

In [4]:
tril = torch.tril(torch.ones(T, T))                 # 1s on/below the diagonal
print(tril)

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


In [5]:
wei = wei.masked_fill(tril == 0, float("-inf"))     # future -> -inf
wei = F.softmax(wei, dim=-1)

print("\nAFTER causal mask — strictly lower-triangular; each token only looks back:")
print(wei[0].round(decimals=2))
out = wei @ v
print("\nrow 0 attends only to token 0:", wei[0, 0].round(decimals=2).tolist())


AFTER causal mask — strictly lower-triangular; each token only looks back:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3500, 0.6500, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5200, 0.4400, 0.0400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1800, 0.1700, 0.2600, 0.3900, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2900, 0.2000, 0.1400, 0.1300, 0.2400, 0.0000, 0.0000, 0.0000],
        [0.1700, 0.1100, 0.2800, 0.1100, 0.1800, 0.1600, 0.0000, 0.0000],
        [0.1200, 0.1200, 0.2800, 0.0900, 0.1200, 0.0700, 0.2100, 0.0000],
        [0.0800, 0.2600, 0.0800, 0.0900, 0.2300, 0.0900, 0.1000, 0.0600]],
       grad_fn=<RoundBackward1>)

row 0 attends only to token 0: [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## Reference check against the course output
Both matrices printed above are the same as the course's, digit for digit (torch 2.15.0). I also check the
behaviour directly: nothing above the diagonal, every row still sums to 1, and the last row (token 7 can see
everything) is unchanged by the mask.

In [6]:
course_after = torch.tensor([
    [1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.35, 0.65, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.52, 0.44, 0.04, 0.00, 0.00, 0.00, 0.00, 0.00],
    [0.18, 0.17, 0.26, 0.39, 0.00, 0.00, 0.00, 0.00],
    [0.29, 0.20, 0.14, 0.13, 0.24, 0.00, 0.00, 0.00],
    [0.17, 0.11, 0.28, 0.11, 0.18, 0.16, 0.00, 0.00],
    [0.12, 0.12, 0.28, 0.09, 0.12, 0.07, 0.21, 0.00],
    [0.08, 0.26, 0.08, 0.09, 0.23, 0.09, 0.10, 0.06]])
mine = wei[0].detach().round(decimals=2)
print("max |mine - course|        :", (mine - course_after).abs().max().item())
print("all future weights are 0   :", bool((torch.triu(wei, 1) == 0).all()))
print("every row still sums to 1  :", bool(torch.allclose(wei.sum(-1), torch.ones(B, T))))
print("output shape                :", tuple(out.shape))

max |mine - course|        : 0.0
all future weights are 0   : True
every row still sums to 1  : True
output shape                : (4, 8, 16)


## Test yourself

**1. Why set future scores to `-inf` instead of just multiplying the weights by 0 after softmax?**
Masking before the softmax means the remaining weights automatically re-normalise to sum to 1. Zeroing after
softmax would leave rows that sum to less than 1.

**2. Why is the last row the same before and after the mask?**
The last token is allowed to see every earlier position, so nothing in its row gets masked — the softmax is the same.

**3. Why does a GPT need this mask at all?**
It's trained to predict the next token at every position. Without the mask, position t could read t+1 directly and
the loss would fall without the model learning anything usable for generation, where the future doesn't exist yet.

## My notes
- The whole "decoder" idea is one `masked_fill` with a lower-triangular matrix before the softmax.
- Surprising: row 0 is always exactly `[1, 0, …, 0]` — the first token can only attend to itself.